# SupeRR Selector — player ratings

This notebook reads the silver files and builds two lists: batters and bowlers.

Each player has a baseline from their usual rates, then three pressure bands, a within-match comparison, a clutch term, form, and consistency. A season they missed counts as zero. Every career rate is divided by the same five-season total, 4.0951. The weights are 0.6561, 0.729, 0.81, 0.9, and 1 for 2022 through 2026.

The calculations live in `player_ratings.py`. This notebook runs them and shows the tables. The silver files are not changed.

## Load and tag balls

A legal ball is not a wide, a no-ball, or a super over. A wide or a no-ball adds one conceded run. Bye and leg-bye amounts are not in the file, and neither are extra runs beyond that one-run penalty, so a team total is still a lower bound. An ordinary chase target is the first-innings total plus one. A DLS target is used as supplied.

A legal ball is not a wide, a no-ball, or a super over. Situation points come from the phase of the innings, wickets down, balls left in a chase, required rate, a close chase, a knockout, a final, and rain. Zero is 0–2 points, medium is 3–5, and high is 6 or more.

In [1]:
from pathlib import Path

import pandas as pd

import player_ratings as ratings

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 30)

balls, matches, comps = ratings.load()
work, legal = ratings.add_situation(balls)
print("deliveries", len(balls), "legal, excluding super overs", len(legal))
print("five-season weight total", round(ratings.W_SUM, 4))
bands = ratings.band_summary(legal)
display(bands.assign(share=lambda frame: (100 * frame["share"]).round(1)))

deliveries 200999 legal, excluding super overs 192675
five-season weight total 4.0951


,balls,share
band,,
zero,129686,67.3
medium,50915,26.4
high,12074,6.3


## Elo

Team strength updates after each match. Batting strength and bowling strength update after each legal ball, on separate IPL and international ladders. Venue strength is fit on the whole history of the ground and applied back to every season. A ball is worth the runs and the wicket chance minus what those ratings expected.

In [2]:
mu, p0, rho = ratings.expectation_tables(legal)
venues = ratings.venue_ratings(legal, rho)
legal = ratings.fit_elo(legal, mu, p0, venues)
summary = legal.attrs["elo_summary"]
for key, value in summary.items():
    if isinstance(value, str):
        print(key + ":", value)
    else:
        print(key + ":", round(value, 1))
print("grounds", len(venues), "mean bat runs per legal ball", round(rho, 3))

k_run: 4.0
k_wicket: 16.0
k_team: 20.0
batter_elo_min: 948.8
batter_elo_max: 2516.3
batter_elo_sd: 140.8
bowler_elo_min: 1051.7
bowler_elo_max: 2094.2
bowler_elo_sd: 120.0
team_elo_min: 1361.3
team_elo_max: 1647.0
step_sizes: as written
grounds 97 mean bat runs per legal ball 1.394


## Batter scores

Baseline is strike rate and average against the phases that batter actually plays. The three bands are the strength-adjusted ball value inside zero, medium, and high pressure. Peers compare an innings with teammates and the opposition. Clutch is that comparison in the high band. Form uses up to eight earlier innings. Consistency prefers a narrower spread of match values.

Each term is a z-score on players with at least 30 decay-weighted legal balls, and each z-score stops at 3. The composite is 0.15 baseline, 0.10 zero, 0.15 medium, 0.20 high, 0.15 clutch, 0.10 peers, 0.05 form, and 0.05 consistency.

In [3]:
batters = ratings.score_batters(legal)
print("batters", len(batters), "ranked", int(batters["ranked"].sum()))
batter_cols = [
    "rank", "player_name", "role", "composite", "baseline",
    "band_zero", "band_medium", "band_high", "clutch", "peers", "form", "consistency",
    "runs", "strike_rate", "average", "n_eff", "high_n_eff",
]
display(ratings.top_table(batters, batter_cols))
print("roles in the batter top 10")
print(batters.loc[batters["rank"].le(10), "role"].value_counts().to_string())

batters 888 ranked 70


,rank,player_name,role,composite,baseline,band_zero,band_medium,band_high,clutch,peers,form,consistency,runs,strike_rate,average,n_eff,high_n_eff
0,1.0,Rovman Powell,middle order,1.016,0.786,-0.239,0.415,2.313,3.000,0.214,-0.127,-1.367,2287,147.074,28.235,1242.328,144.888
1,2.0,Tim David,middle order,0.990,2.211,0.487,1.618,-0.101,2.097,0.869,0.623,-0.901,2153,177.202,34.726,1002.709,101.644
2,3.0,Mohammad Nawaz,lower order,0.871,0.094,-0.410,1.185,0.444,3.000,1.309,1.448,-0.446,705,142.424,21.364,405.336,67.537
3,4.0,David Miller,middle order,0.870,1.585,-0.967,0.418,1.642,3.000,-0.245,-1.276,-0.485,1902,149.060,38.816,977.315,80.510
4,5.0,Naman Dhir,top order,0.859,1.372,-0.282,1.311,2.368,-0.279,1.378,0.395,-2.085,992,168.707,28.343,551.980,54.880
5,6.0,Jos Buttler,top order,0.784,2.064,0.869,0.939,0.714,-0.216,1.523,-0.122,-0.198,4384,155.461,41.752,2303.778,37.050
6,7.0,Rinku Singh,middle order,0.750,1.689,-0.919,0.733,3.000,-0.216,0.658,-1.527,-1.568,1949,152.623,38.980,1047.970,97.546
7,8.0,Dinesh Karthik,middle order,0.569,1.482,-0.868,0.743,0.351,1.598,1.264,0.050,-2.326,1061,166.562,31.206,452.142,62.200
8,9.0,Faheem Ashraf,lower order,0.543,-0.203,0.186,1.455,0.659,1.174,0.946,-0.366,-0.954,295,144.608,15.526,177.631,46.206
9,10.0,Mahendra Singh Dhoni,lower order,0.531,0.923,0.087,0.562,1.603,-0.216,1.267,0.101,-2.406,690,150.000,31.364,352.416,122.564


roles in the batter top 10
role
middle order    5
lower order     3
top order       2


## Bowler scores

The same shape. Baseline uses economy and balls per wicket, against the usual rate for that bowler's pace or spin and the phases they bowl. A lower rate is a higher score. The high band, clutch, and peers use the same direction.

In [4]:
bowlers = ratings.score_bowlers(legal, work)
print("bowlers", len(bowlers), "ranked", int(bowlers["ranked"].sum()))
bowler_cols = [
    "rank", "player_name", "role", "composite", "baseline",
    "band_zero", "band_medium", "band_high", "clutch", "peers", "form", "consistency",
    "wickets", "economy", "balls_per_wicket", "n_eff", "high_n_eff",
]
display(ratings.top_table(bowlers, bowler_cols))
print("roles in the bowler top 10")
print(bowlers.loc[bowlers["rank"].le(10), "role"].value_counts().to_string())

bowlers 647 ranked 103


,rank,player_name,role,composite,baseline,band_zero,band_medium,band_high,clutch,peers,form,consistency,wickets,economy,balls_per_wicket,n_eff,high_n_eff
0,1.0,Jasprit Bumrah,Pace,1.416,1.256,1.907,1.523,1.322,1.447,2.867,0.841,-0.036,85.0,6.920,20.106,1455.083,124.378
1,2.0,Ahmed Daniyal,Pace,1.359,1.590,1.479,1.381,3.000,-0.385,1.881,-0.002,0.712,10.0,5.018,17.100,152.010,44.730
2,3.0,Sunil Narine,Spin,1.338,0.441,0.335,1.251,3.000,0.595,2.842,0.254,1.285,64.0,6.773,23.906,1246.702,32.726
3,4.0,Abrar Ahmed,Spin,1.110,1.332,0.168,1.356,1.672,1.438,1.525,-0.349,0.099,46.0,6.565,16.609,689.250,54.600
4,5.0,Jofra Archer,Pace,1.050,0.267,0.532,0.860,2.847,0.534,0.998,0.969,0.597,71.0,8.636,17.761,1144.803,72.576
5,6.0,Rashid Khan,Spin,1.044,0.691,0.173,0.788,1.065,2.696,1.542,0.684,-0.014,153.0,7.332,17.954,2235.578,130.439
6,7.0,Mitchell Starc,Pace,1.043,0.163,0.284,1.328,2.061,3.000,-0.277,-0.758,-0.118,53.0,9.194,15.170,672.678,49.423
7,8.0,Sufiyan Muqeem,Spin,1.026,1.942,-0.777,1.542,0.499,2.279,1.705,0.630,-1.257,39.0,5.919,13.282,449.880,57.600
8,9.0,Krunal Pandya,Spin,0.968,0.010,-0.168,0.677,3.000,1.049,1.359,-0.108,-0.124,56.0,7.628,22.446,1045.079,52.657
9,10.0,Jamie Overton,Pace,0.936,0.898,0.926,1.588,1.693,0.898,0.413,0.657,-1.538,36.0,8.253,13.833,470.300,43.200


roles in the bowler top 10
role
Pace    5
Spin    5


## Rankings

A player is ranked in a skill with at least 120 decay-weighted legal balls, and at least 30 of those in the high band. Anyone under the line stays in the full table with no rank. The two lists are saved under `ratings/`.

In [5]:
out = Path("ratings")
out.mkdir(exist_ok=True)
batters.to_csv(out / "batters.csv", index=False)
bowlers.to_csv(out / "bowlers.csv", index=False)
print("wrote", out / "batters.csv", "and", out / "bowlers.csv")
print("decision: floor is 120 decay-weighted legal balls and 30 high-band balls")
print("decision: season denominator", round(ratings.W_SUM, 4))
print("decision: elo step sizes", summary["step_sizes"])

wrote ratings\batters.csv and ratings\bowlers.csv
decision: floor is 120 decay-weighted legal balls and 30 high-band balls
decision: season denominator 4.0951
decision: elo step sizes as written
